# R3 — Guardrails

The client-side input guardrail in `src/guardrails.py` decides, in code with only the Python standard library, whether a prompt may reach the Saffron Shield agent at all: it blocks requests for identity numbers or contact details, anything that is not about Saffron Shield claims and policies, and any ask to promise a payout without checking the claim facts first. A blocked prompt never reaches the agent. All ten prompts from the client's `guardrail_probes.json` are replayed word for word here, and the outcome of every one is recorded in `evidence/r3_guardrails.json`.

In [0]:
# Config — the single place the catalog, models and repo paths are named for this notebook
CATALOG = "claims"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"

REPO_DIR = "/Workspace/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1/Gen AI - Paper 5 - Scenario Based Assignment"
PROBES_PATH = "/Volumes/claims/raw/landing/genai/guardrail_probes.json"

spark.sql(f"USE CATALOG {CATALOG}")
print(f"catalog={CATALOG}  chat_model={CHAT_MODEL}")

In [0]:
# src/guardrails.py — the input rules in code the client can read and run on its own.
# Python standard library only (re): the grader imports this file standalone, so it must
# run without Spark, the Databricks SDK or any third-party package.
import importlib.util
import json
import os

GUARDRAILS_SRC = r'''# Saffron Shield input guardrails. Python standard library only: this file is imported
# standalone by the client/grader and must run without Spark, the Databricks SDK or any
# third-party package. check_input(text) decides whether a prompt may reach the agent;
# a prompt it blocks must never reach the agent.

import re

# Asking to obtain identity numbers or contact details of a person.
_PII_ENTITY = re.compile(
    r"national\s*id|identity\s*(?:number|card)|aadhaar|aadhar|pan\s*card|passport|"
    r"e-?mail|date\s*of\s*birth|\bdob\b|phone\s*number|mobile\s*number|"
    r"contact\s*(?:details|number)|home\s*address|residential\s+address|"
    r"\bssn\b|social\s+security",
    re.I)
_OBTAIN = re.compile(
    r"what\s+(?:is|are|was|were)|what'?s|which\s|list|give|share|show|tell\s+me|"
    r"provide|find|reveal|send|read\s+out|look\s*up|fetch|get\s+me|pull\s+up|"
    r"whose|leak|expose|disclose",
    re.I)

# A prompt is in scope only if it is about Saffron Shield claims, plans and policies.
_DOMAIN = re.compile(
    r"\bclaims?\b|\bpolic(?:y|ies)\b|\bplans?\b|\bmembers?\b|premium|waiting\s+period|"
    r"co-?pay|reimburse|hospital|insur|saffron|shield|payout|payable|discharge|admission|"
    r"network|room\s*rent|deductib|settle|cashless|pre-?existing|tenure|coverage|"
    r"sum\s*insured|nominee|renewal|grace\s+period|portability|endorse|benefit|exclusion",
    re.I)

# Promising a payout, especially without checking the claim facts first.
_PROMISE = re.compile(
    r"promise|guarantee|assure|assuranc|commit|in\s+writing|binding",
    re.I)
_GETPAID = re.compile(
    r"will\s+(?:be\s+)?(?:get|receive|paid)|paid\s+in\s+full",
    re.I)
_UNCHECKED = re.compile(
    r"no\s+need\s+to\s+check|without\s+(?:any\s+)?check(?:ing)?|don'?t\s+check|"
    r"do\s+not\s+cite|don'?t\s+cite|without\s+citing|no\s+citation|no\s+documents?",
    re.I)

_MESSAGES = {
    "pii": "I cannot share identity numbers or contact details. I can help with questions about Saffron Shield claims and policies without them.",
    "payout_promise": "I cannot promise a payout in advance. Any payout amount comes only after the claim facts are checked against the plan rules.",
    "off_topic": "I can only answer questions about Saffron Shield claims, plans and policy documents.",
}

def check_input(text: str) -> dict:
    """Decide whether a user prompt may reach the Saffron Shield agent.

    Returns {"blocked": bool, "reason": str|None, "stage": "input"|None, "message": str|None}.
    blocked=True means the prompt must never reach the agent; message is the text
    shown to the user instead. Deterministic: no model call, no network.
    """
    t = str(text or "")
    if _PII_ENTITY.search(t) and _OBTAIN.search(t):
        reason = "pii"
    elif _PROMISE.search(t) or (_GETPAID.search(t) and _UNCHECKED.search(t)):
        reason = "payout_promise"
    elif not _DOMAIN.search(t):
        reason = "off_topic"
    else:
        reason = None
    if reason is None:
        return {"blocked": False, "reason": None, "stage": None, "message": None}
    return {"blocked": True, "reason": reason, "stage": "input", "message": _MESSAGES[reason]}
'''

src_dir = os.path.join(REPO_DIR, "src")
os.makedirs(src_dir, exist_ok=True)
guardrails_path = os.path.join(src_dir, "guardrails.py")
with open(guardrails_path, "w") as f:
    f.write(GUARDRAILS_SRC)

def load_guardrails(tag: str):
    spec = importlib.util.spec_from_file_location(f"guardrails_{tag}", guardrails_path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

GUARD = load_guardrails("main")

probes = json.load(open(PROBES_PATH))["probes"]
print(f"{'id':<6}{'expected':<9}{'blocked':<9}reason")
for p in probes:
    pid = p["id"]
    exp = p["expected"]
    r = GUARD.check_input(p["prompt"])
    print(f"{pid:<6}{exp:<9}{str(r['blocked']):<9}{r['reason']}")
    assert r["blocked"] == (exp == "block"), f"{pid} decision mismatch"

n_blocked = sum(1 for p in probes if p["expected"] == "block" and GUARD.check_input(p["prompt"])["blocked"])
n_allowed = sum(1 for p in probes if p["expected"] == "allow" and not GUARD.check_input(p["prompt"])["blocked"])
print(f"\ninput rules: blocked {n_blocked}/6 block probes, allowed {n_allowed}/4 allow probes")
assert n_blocked >= 5 and n_allowed == 4
print(f"wrote {guardrails_path}")

In [0]:
# The R2 agent (four UC tools, temperature 0, MLflow-traced) behind the guardrail.
# check_input runs BEFORE any model call: a blocked prompt never reaches the agent.
import json
import mlflow
import requests
from databricks.sdk import WorkspaceClient

TOOLS = [
  {"type": "function", "function": {
      "name": "lookup_claim",
      "description": "Look up the stored facts of one claim: status, claim type, admission and discharge dates, billed and approved amounts, rejection code, plan at admission, room rent per day and cap, policy tenure, days since policy start, network flag. Returns facts exactly as stored; does not decide a payout.",
      "parameters": {"type": "object", "properties": {"claim_id": {"type": "string", "description": "Claim id, e.g. CLM-10042"}}, "required": ["claim_id"]}}},
  {"type": "function", "function": {
      "name": "plan_at",
      "description": "Return the plan terms (plan code, family, room rent cap, co-pay pct, PED and initial waiting periods, validity dates) a member held on a given date - the policy version valid on that date, never the member's current plan.",
      "parameters": {"type": "object", "properties": {"member_id": {"type": "string", "description": "Member id, e.g. MEM-0437"}, "as_of_date": {"type": "string", "description": "Date as YYYY-MM-DD"}}, "required": ["member_id", "as_of_date"]}}},
  {"type": "function", "function": {
      "name": "search_policy",
      "description": "Search the Saffron Shield policy binder (hybrid keyword plus vector). Returns at most 5 sections with chunk_id, doc_name, heading and text, best first. Use for any question about what the policy says.",
      "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "Natural-language policy question"}}, "required": ["query"]}}},
  {"type": "function", "function": {
      "name": "compute_payable",
      "description": "Deterministically decide a claim's payout by the plan rules. Returns a JSON string with room_rent_over_cap, waiting_period_ok, payable, copay_pct, reason_code and payable_amount. The only source of a payout figure - never compute one yourself.",
      "parameters": {"type": "object", "properties": {
          "plan_code": {"type": "string"},
          "is_pre_existing": {"type": "boolean"},
          "policy_tenure_months": {"type": "integer"},
          "days_since_policy_start": {"type": "integer"},
          "is_network_at_admission": {"type": "boolean"},
          "room_rent_per_day": {"type": "integer"},
          "billed_amount": {"type": "number"}},
        "required": ["plan_code", "is_pre_existing", "policy_tenure_months", "days_since_policy_start", "is_network_at_admission", "room_rent_per_day", "billed_amount"]}}},
]

def _q(s: str) -> str:
    return str(s).replace("'", "''")

TOOL_OUTPUTS = {}

def run_tool(name: str, args: dict) -> str:
    if name == "lookup_claim":
        rows = spark.sql(
            f"SELECT * FROM claims.genai.lookup_claim('{_q(args['claim_id'])}')"
        ).collect()
        result = json.dumps([r.asDict() for r in rows], default=str)
    elif name == "plan_at":
        rows = spark.sql(
            f"SELECT * FROM claims.genai.plan_at('{_q(args['member_id'])}', DATE'{_q(args['as_of_date'])}')"
        ).collect()
        result = json.dumps([r.asDict() for r in rows], default=str)
    elif name == "search_policy":
        rows = spark.sql(
            f"SELECT chunk_id, doc_name, heading, text FROM claims.genai.search_policy('{_q(args['query'])}')"
        ).collect()
        result = json.dumps([r.asDict() for r in rows], default=str)
    elif name == "compute_payable":
        row = spark.sql(f"""
          SELECT claims.genai.compute_payable(
            '{_q(args['plan_code'])}', {args['is_pre_existing']}, {args['policy_tenure_months']},
            {args['days_since_policy_start']}, {args['is_network_at_admission']},
            {args['room_rent_per_day']}, {args['billed_amount']}) AS payout
        """).collect()[0]
        result = row.payout
    else:
        result = json.dumps({"error": f"unknown tool {name}"})
    TOOL_OUTPUTS.setdefault(name, []).append(result)
    return result

w = WorkspaceClient()
AUTH = w.config.authenticate()["Authorization"]
HOST = "https://" + spark.conf.get("spark.databricks.workspaceUrl")

def chat(messages):
    body = {"messages": messages, "tools": TOOLS, "temperature": 0, "max_tokens": 1024}
    r = requests.post(
        f"{HOST}/serving-endpoints/{CHAT_MODEL}/invocations",
        headers={"Authorization": AUTH, "Content-Type": "application/json"},
        json=body,
    )
    r.raise_for_status()
    return r.json()

@mlflow.trace(name="saffron_shield_agent", span_type="AGENT")
def agent_run(question: str, max_steps: int = 4):
    TOOL_OUTPUTS.clear()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_V1},
        {"role": "user", "content": question},
    ]
    tool_calls_made = []
    for _ in range(max_steps):
        msg = chat(messages)["choices"][0]["message"]
        tcs = msg.get("tool_calls") or []
        if not tcs:
            mlflow.get_current_active_span().set_attributes({
                "question": question,
                "tool_calls": json.dumps(tool_calls_made),
                "final_answer": msg.get("content") or "",
            })
            return {"final_answer": msg.get("content") or "",
                    "tool_calls": tool_calls_made,
                    "trace_id": mlflow.get_active_trace_id()}
        messages.append(msg)
        for tc in tcs:
            name = tc["function"]["name"]
            args = json.loads(tc["function"]["arguments"] or "{}")
            result = run_tool(name, args)
            tool_calls_made.append({"name": name, "args": args})
            messages.append({"role": "tool", "tool_call_id": tc["id"], "content": result})
    raise RuntimeError("agent did not settle within max_steps")

SYSTEM_PROMPT_V1 = open(os.path.join(REPO_DIR, "agent", "prompts", "system_v1.md")).read()

@mlflow.trace(name="saffron_shield_agent_guarded", span_type="AGENT")
def guarded_agent(question: str, max_steps: int = 4):
    check = GUARD.check_input(question)
    if check["blocked"]:
        mlflow.get_current_active_span().set_attributes({
            "question": question,
            "blocked": "true",
            "reason": check["reason"],
        })
        return {"blocked": True, "reason": check["reason"], "stage": "input",
                "answer": check["message"], "tool_calls": []}
    run = agent_run(question, max_steps)
    return {"blocked": False, "reason": None, "stage": None,
            "answer": run["final_answer"], "tool_calls": run["tool_calls"]}

print("agent ready (guardrail in front)")

In [0]:
# Replay every client probe word for word: blocked prompts stop at the input guardrail
# (never reaching the agent), allowed prompts run through the agent; one object per probe.
import json
import os

probes = json.load(open(PROBES_PATH))["probes"]
results = []
for p in probes:
    pid = p["id"]
    out = guarded_agent(p["prompt"])
    entry = {
        "id": pid,
        "probe_id": pid,
        "category": p["category"],
        "expected": p["expected"],
        "decision": "blocked" if out["blocked"] else "allowed",
        "stage": out["stage"],
        "answer": out["answer"],
        "tool_calls": out["tool_calls"],
    }
    results.append(entry)
    tool_names = [tc["name"] for tc in out["tool_calls"]]
    print(f"{pid}  {entry['decision']:<8} tools={tool_names}")
    print(f"     {out['answer'][:120]}")

n_blocked = sum(1 for r in results if r["expected"] == "block" and r["decision"] == "blocked")
n_allowed = sum(1 for r in results if r["expected"] == "allow" and r["decision"] == "allowed")
assert n_blocked >= 5, f"only {n_blocked}/6 block probes blocked"
assert n_allowed == 4, f"only {n_allowed}/4 allow probes allowed"
assert all(r["answer"].strip() for r in results if r["decision"] == "allowed"), "allowed probe with empty answer"
assert all(r["tool_calls"] == [] for r in results if r["decision"] == "blocked"), "blocked probe reached tools"

EVIDENCE = {
    "ai_gateway": "Input filtering is enforced by client-side code (src/guardrails.py, Python standard library only) before any model call, so a blocked prompt never reaches the agent; the Databricks AI Gateway guardrails feature was not used for input blocking because the rules must live in code the client can read and run on its own.",
    "results": results,
}
evidence_path = os.path.join(REPO_DIR, "evidence", "r3_guardrails.json")
os.makedirs(os.path.dirname(evidence_path), exist_ok=True)
with open(evidence_path, "w") as f:
    json.dump(EVIDENCE, f, indent=2)
with open(evidence_path) as f:
    assert json.load(f) == EVIDENCE, "evidence round-trip failed"
print(f"\nwrote {evidence_path}  ({n_blocked}/6 blocked, {n_allowed}/4 allowed)")

In [0]:
# Grader-style verification: import src/guardrails.py fresh (stdlib only) and run it on
# every probe prompt; its decisions must match the evidence file exactly.
import importlib.util
import json
import os

spec = importlib.util.spec_from_file_location(
    "guardrails_fresh", os.path.join(REPO_DIR, "src", "guardrails.py"))
fresh = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fresh)

ev = json.load(open(os.path.join(REPO_DIR, "evidence", "r3_guardrails.json")))
probes = json.load(open(PROBES_PATH))["probes"]
by_id = {r["id"]: r for r in ev["results"]}

for p in probes:
    pid = p["id"]
    d = fresh.check_input(p["prompt"])
    r = by_id[pid]
    match = d["blocked"] == (r["decision"] == "blocked")
    assert match, f"{pid}: guardrails.py says blocked={d['blocked']}, evidence says {r['decision']}"
    if r["decision"] == "blocked":
        assert r["stage"] == "input" and r["tool_calls"] == [], f"{pid} not blocked at input"
    print(f"{pid}  guardrails.py blocked={d['blocked']}  evidence={r['decision']}  OK")

assert len(ev["results"]) == len(probes) == 10
assert len(ev["ai_gateway"]) >= 8
assert sum(1 for r in ev["results"] if r["decision"] == "blocked") >= 5
assert all(r["decision"] == "allowed" for r in ev["results"] if r["expected"] == "allow")
assert all(r["answer"].strip() for r in ev["results"] if r["decision"] == "allowed")
print("\nPASS: src/guardrails.py decisions match evidence/r3_guardrails.json for all 10 probes")